# MiniGPT tr?n Google Colab: train v? xu?t model d?ng ???c

Notebook n?y t? clone project v?o `/content`, c?i dependencies, hu?n luy?n model v? l?u checkpoint v?o Google Drive. Lu?ng g?m pretraining, SFT/DPO t?y ch?n, r?i n?p checkpoint ?? sinh v?n b?n. Tokenizer l? GPT-2 BPE (`tiktoken`).

**B?t ??u:** m? notebook trong Google Colab, ch?n **Runtime ? Change runtime type ? T4 GPU** (ho?c GPU kh? d?ng), sau ?? ch?y cell t? tr?n xu?ng. L?n ??u c?n Internet ?? clone m? ngu?n/t?i d? li?u v? c?n c?p quy?n mount Google Drive. Checkpoint l?u tr?n Drive n?n c?n sau khi Colab ng?t runtime.

Th?ng s? m?c ??nh nh? ?? ki?m tra lu?ng tr?n GPU Colab; v?i tr?m b??c ch?a ?? ?? c? ch?t l??ng h?i tho?i t?t. D?ng corpus c?a b?n v? t?ng `MAX_STEPS` ?? hu?n luy?n l?u h?n.

In [ ]:
# Bootstrap Colab: clone project v? chuy?n working directory v?o checkout.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ulyssesllc/ulysses-gpt.git"
ROOT = Path("/content/ulysses-gpt")
if not (ROOT / "train.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
assert (ROOT / "train.py").exists(), f"Kh?ng t?m th?y train.py trong {ROOT}"
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


## L?u model l?u d?i tr?n Google Drive

Colab c? th? x?a d? li?u trong `/content` khi runtime k?t th?c. Mount Drive ?? gi? checkpoint v? model ?? fine-tune.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/mini-gpt-scratch')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Model artifacts will be saved to:', DRIVE_OUTPUT_DIR)


## 1. C?i ??t dependencies v? ki?m tra GPU

Project ?? ???c clone v? notebook ?ang ch?y t? th? m?c repo tr?n Colab. C?i package t? checkout n?y. N?u PyTorch/GPU kh?ng ???c nh?n di?n, ki?m tra runtime type r?i ch?n **Runtime ? Restart session**.

In [ ]:
%pip install -e .


In [ ]:
import torch
from mini_gpt.dataset import BPETokenizer
from mini_gpt.model import GPTConfig, MiniGPT

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
tokenizer = BPETokenizer("gpt2")
print("Tokenizer vocabulary:", tokenizer.vocab_size)


## 2. C?u h?nh pipeline

M?c ??nh pretraining d?ng Tiny Shakespeare m? `train.py` t? t?i. C? th? d?ng file corpus ?? upload v?o `/content` b?ng `PRETRAIN_FILE`, ho?c Hugging Face b?ng `PRETRAIN_HF_DATASET`. Khi b?t SFT/DPO, nh?p dataset ID v? t?n c?t ? c?u h?nh. SFT/DPO c?n d? li?u Hugging Face v? checkpoint pretrain.

Checkpoint ???c ghi v?o Google Drive. Gi?m `MAX_STEPS`, `MAX_TOKENS`, `BATCH_SIZE` cho l?n ch?y th?; t?ng l?n cho l?n hu?n luy?n th?t. Ch?n model nh? n?u GPU Colab c? ?t VRAM; `N_EMBD` ph?i chia h?t cho `N_HEAD`.

In [ ]:
# Dataset pretraining: ch?n t?i ?a m?t ngu?n.
PRETRAIN_FILE = None                 # V? d?: "/content/corpus.txt"
PRETRAIN_HF_DATASET = None           # V? d?: "Salesforce/wikitext"
PRETRAIN_HF_CONFIG = None            # V? d?: "wikitext-2-raw-v1"
PRETRAIN_HF_SPLIT = "train"
TEXT_COLUMN = "auto"

# Model/train. C?c gi? tr? n?y kh?i ??u nh? ?? ph? h?p GPU Colab ph? th?ng.
BLOCK_SIZE = 128
N_LAYER = 4
N_HEAD = 4
N_EMBD = 128
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
EPOCHS = 1
MAX_STEPS = 300
MAX_TOKENS = 200_000
MAX_VAL_TOKENS = 10_000
LEARNING_RATE = 3e-4
SEED = 42
CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / "checkpoints"

# Fine-tuning t?y ch?n. B?t pha c?n ch?y v? nh?p ??ng dataset/c?t.
RUN_SFT = False
RUN_DPO = False
POST_HF_DATASET = None              # V? d?: "your-org/instruction-data"
POST_HF_CONFIG = None
POST_HF_SPLIT = "train"
PROMPT_COLUMN = "prompt"
RESPONSE_COLUMN = "response"
CHOSEN_COLUMN = "chosen"
REJECTED_COLUMN = "rejected"
POST_MAX_SAMPLES = 2_000
POST_EPOCHS = 1
SAMPLE_PROMPT = "Once upon a time"
print("Device:", DEVICE)
print("Checkpoint folder:", CHECKPOINT_DIR)


## 3. Pretraining

Cell n?y g?i `train.py`: l?m s?ch/encode d? li?u, t?o validation split, train causal GPT, in loss/perplexity/accuracy v? l?u checkpoint ho?n ch?nh l?n Drive. C? th? resume m?t checkpoint ? `RESUME_FROM`.

In [ ]:
import subprocess

def run_train(args):
    command = [sys.executable, str(ROOT / "train.py"), *map(str, args)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=ROOT, check=True)

pretrain_args = [
    "--task", "pretrain",
    "--block-size", BLOCK_SIZE, "--batch-size", BATCH_SIZE,
    "--grad-accum-steps", GRAD_ACCUM_STEPS, "--epochs", EPOCHS,
    "--max-steps", MAX_STEPS, "--max-tokens", MAX_TOKENS,
    "--max-val-tokens", MAX_VAL_TOKENS, "--learning-rate", LEARNING_RATE,
    "--n-layer", N_LAYER, "--n-head", N_HEAD, "--n-embd", N_EMBD,
    "--checkpoint-dir", CHECKPOINT_DIR, "--seed", SEED, "--prompt", SAMPLE_PROMPT,
]
if PRETRAIN_FILE:
    pretrain_args += ["--dataset", PRETRAIN_FILE]
if PRETRAIN_HF_DATASET:
    pretrain_args += ["--hf-dataset", PRETRAIN_HF_DATASET, "--hf-split", PRETRAIN_HF_SPLIT, "--text-column", TEXT_COLUMN]
    if PRETRAIN_HF_CONFIG:
        pretrain_args += ["--hf-config", PRETRAIN_HF_CONFIG]
# Resume n?u c? checkpoint ?? l?u: RESUME_FROM = CHECKPOINT_DIR / "checkpoint_step_100.pt"
RESUME_FROM = None
if RESUME_FROM:
    pretrain_args += ["--resume", RESUME_FROM]
run_train(pretrain_args)
PRETRAIN_CHECKPOINT = CHECKPOINT_DIR / "final_minigpt.pt"
assert PRETRAIN_CHECKPOINT.exists(), f"Kh?ng t?o ???c checkpoint: {PRETRAIN_CHECKPOINT}"
print("Pretrained model:", PRETRAIN_CHECKPOINT)


## 4. SFT t?y ch?n

SFT d?ng c?c c?t prompt/response. B?t `RUN_SFT=True` v? c?u h?nh `POST_HF_DATASET` c?ng t?n c?t ? ph?n c?u h?nh. Model sau SFT s? tr? th?nh checkpoint ??u v?o cho DPO (n?u b?t) v? model cu?i.

In [ ]:
SFT_CHECKPOINT = CHECKPOINT_DIR / "sft_minigpt.pt"
if RUN_SFT:
    if not POST_HF_DATASET:
        raise ValueError("RUN_SFT=True y?u c?u POST_HF_DATASET")
    sft_args = [
        "--task", "sft", "--hf-dataset", POST_HF_DATASET,
        "--hf-split", POST_HF_SPLIT, "--prompt-column", PROMPT_COLUMN,
        "--response-column", RESPONSE_COLUMN, "--checkpoint", PRETRAIN_CHECKPOINT,
        "--checkpoint-dir", CHECKPOINT_DIR, "--block-size", BLOCK_SIZE,
        "--batch-size", BATCH_SIZE, "--epochs", POST_EPOCHS,
        "--max-samples", POST_MAX_SAMPLES, "--learning-rate", LEARNING_RATE,
        "--seed", SEED, "--prompt", SAMPLE_PROMPT,
    ]
    if POST_HF_CONFIG:
        sft_args += ["--hf-config", POST_HF_CONFIG]
    run_train(sft_args)
    assert SFT_CHECKPOINT.exists(), f"Kh?ng t?o ???c checkpoint: {SFT_CHECKPOINT}"
    print("SFT model:", SFT_CHECKPOINT)


## 5. DPO t?y ch?n

DPO d?ng c?c c?t prompt/chosen/rejected. B?t `RUN_DPO=True` v? c?u h?nh d? li?u preference. N?u SFT ?? ch?y, DPO kh?i t?o t? model SFT; n?u kh?ng, kh?i t?o t? model pretrain. DPO l?u model ?? c?n ch?nh v?o `dpo_minigpt.pt`.

In [ ]:
DPO_CHECKPOINT = CHECKPOINT_DIR / "dpo_minigpt.pt"
if RUN_DPO:
    if not POST_HF_DATASET:
        raise ValueError("RUN_DPO=True y?u c?u POST_HF_DATASET")
    dpo_init = SFT_CHECKPOINT if RUN_SFT else PRETRAIN_CHECKPOINT
    dpo_args = [
        "--task", "dpo", "--hf-dataset", POST_HF_DATASET,
        "--hf-split", POST_HF_SPLIT, "--prompt-column", PROMPT_COLUMN,
        "--chosen-column", CHOSEN_COLUMN, "--rejected-column", REJECTED_COLUMN,
        "--checkpoint", dpo_init, "--checkpoint-dir", CHECKPOINT_DIR,
        "--block-size", BLOCK_SIZE, "--batch-size", BATCH_SIZE,
        "--epochs", POST_EPOCHS, "--max-samples", POST_MAX_SAMPLES,
        "--learning-rate", LEARNING_RATE, "--seed", SEED, "--prompt", SAMPLE_PROMPT,
    ]
    if POST_HF_CONFIG:
        dpo_args += ["--hf-config", POST_HF_CONFIG]
    run_train(dpo_args)
    assert DPO_CHECKPOINT.exists(), f"Kh?ng t?o ???c checkpoint: {DPO_CHECKPOINT}"
    print("DPO model:", DPO_CHECKPOINT)


## 6. Ch?n model cu?i v? sinh th?

`FINAL_MODEL_PATH` l? file mang ?i s? d?ng. Checkpoint ch?a tr?ng s? c?ng c?u h?nh ki?n tr?c; GPT-2 BPE ???c t?i b?ng `tiktoken`. Khi d?ng ngo?i notebook, c?i package v? n?p checkpoint theo cell m?u n?y.

In [ ]:
if RUN_DPO:
    FINAL_MODEL_PATH = DPO_CHECKPOINT
elif RUN_SFT:
    FINAL_MODEL_PATH = SFT_CHECKPOINT
else:
    FINAL_MODEL_PATH = PRETRAIN_CHECKPOINT

checkpoint = torch.load(FINAL_MODEL_PATH, map_location="cpu", weights_only=False)
model_config = checkpoint.get("model_config")
if model_config is None:
    raise ValueError("Checkpoint thi?u model_config v? kh?ng th? t? d?ng model")
model = MiniGPT(model_config)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()

@torch.inference_mode()
def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    from mini_gpt.generate import generate
    if temperature <= 0:
        raise ValueError("temperature ph?i l?n h?n 0")
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
    out = generate(model, ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0].tolist())

print("Final model:", FINAL_MODEL_PATH.resolve())
print(generate_text(SAMPLE_PROMPT))


## 7. D?ng model trong m?t script kh?c

Sau khi t?i/c?i package n?y, n?p checkpoint nh? sau. Gi? `model.eval()` khi suy lu?n v? ch?n `map_location` theo thi?t b?.

In [ ]:
# V? d? ??c l?p (???ng d?n checkpoint t??ng ??i v?i th? m?c ch?y script):
# import torch
# from mini_gpt.model import MiniGPT
# from mini_gpt.dataset import BPETokenizer
# from mini_gpt.generate import generate
#
# ckpt = torch.load("/content/drive/MyDrive/mini-gpt-scratch/checkpoints/final_minigpt.pt", map_location="cpu", weights_only=False)
# model = MiniGPT(ckpt["model_config"])
# model.load_state_dict(ckpt["model_state_dict"])
# model.eval()
# tokenizer = BPETokenizer("gpt2")
# prompt = torch.tensor([tokenizer.encode("Hello")], dtype=torch.long)
# result = generate(model, prompt, max_new_tokens=80, temperature=0.8, top_k=40)
# print(tokenizer.decode(result[0].tolist()))


### L?u ? khi s? d?ng

- Checkpoint l? model n?n t?ng ?? sinh ti?p v?n b?n; mu?n chatbot h?i tho?i h?u ?ch, h?y pretrain tr?n corpus ph? h?p r?i fine-tune SFT v?i d? li?u prompt/response ch?t l??ng.
- DPO c?n preference pairs t?t v? n?n ch?y sau SFT.
- File checkpoint c?a d? ?n l?u `GPTConfig` d??i d?ng ??i t??ng Python. Khi n?p b?ng PyTorch m?i, d?ng `weights_only=False` v? ch? n?p checkpoint tin c?y.
- `block_size` gi?i h?n s? token ng? c?nh. Prompt d?i h?n gi?i h?n s? ???c c?t ? ph?a tr?i trong qu? tr?nh generate.